# SRQ-FLY D2 — exact FLY state-matched falsification control
Run every cell in order on a Colab T4 GPU. D2 evaluates only exact FLY-4518 at the observed SRQ state budget. It is train-only, uses the locked D1 result, and never opens `test.pt`.

In [ ]:
# === Edit repository/Drive paths only. Do not edit dimension, seed, or gates. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/srq-fly-d2-state-match'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_TRAIN_CACHE = f'{DRIVE_ROOT}/imagenetr_train_feature_cache_seed2025'
TRAIN_CACHE_DIR = '/content/imagenetr_train_feature_cache_seed2025'
D1_RESULT_PATH = f'{DRIVE_ROOT}/srq_fly_imagenetr_d1_seed2025/d1_results.json'
DRIVE_WTA_CACHE = f'{DRIVE_ROOT}/srq_fly_wta_h4518_seed2025'
WTA_CACHE_DIR = '/content/srq_fly_wta_h4518_seed2025'
OUTPUT_DIR = f'{DRIVE_ROOT}/srq_fly_imagenetr_d2_state_match_seed2025'
SEED = 2025
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CONFIG_SHA256 = 'e8c630b728f9b5f554fd94e6d450b3db4b2205d0d94a595095fa7ebdddcda197'

In [ ]:
# Runtime setup. chdir first to avoid Colab's deleted-working-directory failure.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
repo_path = Path(WORK_DIR)
if repo_path.exists(): shutil.rmtree(repo_path)
clone = subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_GIT_URL, WORK_DIR], text=True, capture_output=True)
print(clone.stdout, clone.stderr, sep='')
assert clone.returncode == 0, f'Clone failed ({clone.returncode}). Confirm the D2 branch was pushed.'
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
config_path = Path('configs/srq_fly_imagenetr_d2_state_match.json')
assert hashlib.sha256(config_path.read_bytes()).hexdigest() == CONFIG_SHA256, 'Locked config identity mismatch.'
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))
print('locked seed:', SEED, '| config SHA-256:', CONFIG_SHA256)

In [ ]:
# Restore the verified train cache, D1 reference, and optional reusable FLY-4518 WTA cache.
drive_train, local_train = Path(DRIVE_TRAIN_CACHE), Path(TRAIN_CACHE_DIR)
for name in ['metadata.json', 'train.pt']:
    source = drive_train/name
    assert source.is_file(), f'Missing required D0/D1 cache file: {source}'
    local_train.mkdir(parents=True, exist_ok=True)
    if not (local_train/name).is_file():
        print('COPY', source, f'{source.stat().st_size/2**20:.1f} MiB', flush=True)
        shutil.copy2(source, local_train/name)
d1_path = Path(D1_RESULT_PATH)
assert d1_path.is_file(), f'Missing locked D1 result: {d1_path}'
metadata = json.loads((local_train/'metadata.json').read_text())
assert metadata['dataset'] == 'ImageNet-R' and metadata['checkpoint_sha256'] == CHECKPOINT_SHA256
assert metadata['feature_dim'] == 768 and metadata['finite'] is True
assert metadata['test_features_materialized'] is False and not (local_train/'test.pt').exists()
drive_wta, local_wta = Path(DRIVE_WTA_CACHE), Path(WTA_CACHE_DIR)
if (drive_wta/'metadata.json').is_file():
    local_wta.mkdir(parents=True, exist_ok=True)
    for source in sorted(drive_wta.iterdir()):
        if not (local_wta/source.name).is_file():
            print('COPY WTA', source.name, f'{source.stat().st_size/2**20:.1f} MiB', flush=True)
            shutil.copy2(source, local_wta/source.name)
print('D2 preflight: PASS | D1 reference:', d1_path, '| test.pt absent')
print('FLY-4518 WTA:', 'restored' if (local_wta/'metadata.json').is_file() else 'will build with live progress')

In [ ]:
# Correctness gate: synthetic data only.
tests = ['tests/test_srq_fly_math.py', 'tests/test_srq_fly_d2_state_match.py']
subprocess.run([sys.executable, '-m', 'pytest', '-q', *tests], check=True)
print('SRQ-FLY D2 correctness gate: PASS')

In [ ]:
# Locked control. WTA CACHE shows cache construction; TASK shows each completed stage.
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
shutil.copy2(config_path, output_path/'locked_config.json')
command = [sys.executable, '-u', 'tools/srq_fly_d2_state_match.py', '--config', str(config_path), '--feature-cache-dir', TRAIN_CACHE_DIR, '--code-cache-dir', WTA_CACHE_DIR, '--d1-result', D1_RESULT_PATH, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden']
print('Starting exact FLY-4518: one state-matched control over 20 tasks.', flush=True)
print('WTA CACHE=construction; TASK=completed task; RESUME=completed control.', flush=True)
started = time.time()
completed = subprocess.run(command)
print(f'Runner elapsed: {(time.time()-started)/60:.1f} minutes', flush=True)
assert completed.returncode == 0, 'D2 failed; send the complete traceback without editing config.'
assert (output_path/'d2_results.json').is_file() and not (local_train/'test.pt').exists()
print('SRQ-FLY D2 process: COMPLETE')

In [ ]:
# Save a newly built WTA cache, summarize, download evidence, then STOP.
if (local_wta/'metadata.json').is_file() and not (drive_wta/'metadata.json').is_file():
    assert not drive_wta.exists(), f'Incomplete Drive WTA cache exists: {drive_wta}'
    drive_wta.mkdir(parents=True)
    for source in sorted(local_wta.iterdir()):
        print('SAVE WTA', source.name, f'{source.stat().st_size/2**20:.1f} MiB', flush=True)
        shutil.copy2(source, drive_wta/source.name)
result = json.loads((output_path/'d2_results.json').read_text())
control = result['state_matched_exact_fly']
reference = result['d1_reference']
print('exact FLY-4518 validation AA:', control['validation_average_accuracy'])
print('SRQ validation AA:', reference['srq_validation_average_accuracy'])
print('exact/SRQ state bytes:', control['persistent_state_bytes'], '/', reference['srq_persistent_state_bytes'])
print('comparison:', json.dumps(result['comparison'], indent=2))
print('decision:', result['status'])
print('gates:', json.dumps(result['gates'], indent=2))
archive = shutil.make_archive('/content/srq_fly_imagenetr_d2_state_match', 'zip', root_dir=OUTPUT_DIR)
print('artifact SHA-256:', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
from google.colab import files
files.download(archive)
print('STOP. Send the ZIP for audit; do not evaluate ImageNet-R test.')